# Lab Session 11 — Neural Network & Intro to Deep Learning
## From Perceptron to Deep Network: How Machines Learn Visual Patterns

In [ ]:
# ============================================
# IDENTITAS
# Nama  : [Nama Lengkap]
# NPM   : [NPM]
# Kelas : [Kelas]
# ============================================

## Setup: Load & Eksplorasi Dataset
Dataset **Fashion-MNIST** terdiri dari 60.000 gambar training dan 10.000 gambar test.  
Masing-masing berukuran 28×28 piksel grayscale, dengan 10 kelas.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

# Load Fashion-MNIST
(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

# Label classes
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

# Normalisasi pixel ke [0, 1]
X_train = X_train / 255.0
X_test = X_test / 255.0

# Flatten untuk MLP (28x28 -> 784)
X_train_flat = X_train.reshape(-1, 784)
X_test_flat = X_test.reshape(-1, 784)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Jumlah kelas: {len(class_names)}")
print(f"Kelas: {class_names}")

### Tugas Awal: Visualisasi Dataset
Tampilkan **10 sampel gambar acak** dari dataset beserta labelnya dalam satu figure (2 baris × 5 kolom).

In [ ]:
# TODO: Tampilkan 10 gambar acak dari dataset
# Gunakan plt.subplot(2, 5, i+1) untuk layout 2 baris x 5 kolom
# Jangan lupa: plt.title() untuk label, plt.axis('off'), dan identitas di suptitle

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
indices = np.random.choice(len(X_train), 10, replace=False)

for i, idx in enumerate(indices):
    ax = axes[i // 5, i % 5]
    ax.imshow(X_train[idx], cmap='gray')
    ax.set_title(class_names[y_train[idx]], fontsize=10)
    ax.axis('off')

plt.suptitle('Sampel Fashion-MNIST — [Nama] — [NPM]', fontsize=14)
plt.tight_layout()
plt.show()

---
## 🔬 Eksperimen 1: The Single Neuron — Perceptron from Scratch

**Tujuan:** Memahami mekanisme dasar neuron buatan sebelum menggunakan library.

> *Sebelum Anda menggunakan TensorFlow atau Keras, penting untuk memahami apa yang terjadi di dalam satu neuron. Pada eksperimen ini, Anda akan membuat Perceptron dari nol menggunakan NumPy saja.*

In [ ]:
class Perceptron:
    def __init__(self, n_inputs, learning_rate=0.1):
        self.weights = np.zeros(n_inputs)
        self.bias = 0
        self.lr = learning_rate
        self.history = []  # Menyimpan akurasi per epoch

    def activation(self, z):
        """Step function: return 1 if z >= 0, else 0"""
        # TODO: Lengkapi fungsi aktivasi step function
        # Hint: gunakan conditional atau np.where
        pass

    def predict(self, X):
        """Hitung weighted sum lalu terapkan activation"""
        # TODO: Hitung z = X . weights + bias
        # Lalu return self.activation(z)
        pass

    def train(self, X, y, epochs=20):
        """Latih Perceptron menggunakan Perceptron Learning Rule"""
        for epoch in range(epochs):
            correct = 0
            for xi, yi in zip(X, y):
                pred = self.predict(xi)
                error = yi - pred

                # TODO: Update weights dan bias
                # Perceptron Learning Rule:
                #   w = w + learning_rate * error * x
                #   b = b + learning_rate * error
                pass

                if pred == yi:
                    correct += 1
            acc = correct / len(y)
            self.history.append(acc)
        return self.history

### 1B. Uji pada Logic Gates

In [ ]:
# Dataset Logic Gates
X_and = np.array([[0,0], [0,1], [1,0], [1,1]])
y_and = np.array([0, 0, 0, 1])

X_or = np.array([[0,0], [0,1], [1,0], [1,1]])
y_or = np.array([0, 1, 1, 1])

X_xor = np.array([[0,0], [0,1], [1,0], [1,1]])
y_xor = np.array([0, 1, 1, 0])

print("Datasets siap!")

In [ ]:
def plot_decision_boundary_perceptron(perceptron, X, y, title="Decision Boundary"):
    """Visualisasi decision boundary untuk Perceptron pada data 2D"""
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))
    grid = np.c_[xx.ravel(), yy.ravel()]

    Z = np.array([perceptron.predict(point) for point in grid])
    Z = Z.reshape(xx.shape)

    plt.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
    plt.scatter(X[y==0, 0], X[y==0, 1], c='red', marker='o', s=100, edgecolors='k', label='Kelas 0')
    plt.scatter(X[y==1, 0], X[y==1, 1], c='blue', marker='s', s=100, edgecolors='k', label='Kelas 1')
    plt.title(title)
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.legend()
    plt.grid(True, alpha=0.3)

In [ ]:
# Latih Perceptron pada AND, OR, dan XOR
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

gates = [
    ('AND Gate', X_and, y_and),
    ('OR Gate', X_or, y_or),
    ('XOR Gate', X_xor, y_xor)
]

for i, (name, X, y) in enumerate(gates):
    p = Perceptron(n_inputs=2, learning_rate=0.1)
    history = p.train(X, y, epochs=20)

    # Print hasil prediksi
    print(f"\n{'='*30}")
    print(f"{name}")
    print(f"{'='*30}")
    for xi, yi in zip(X, y):
        pred = p.predict(xi)
        status = '✓' if pred == yi else '✗'
        print(f"  Input: {xi} | Target: {yi} | Prediksi: {pred} | {status}")
    print(f"  Akurasi Akhir: {history[-1]*100:.0f}%")

    # Plot decision boundary
    plt.sca(axes[i])
    plot_decision_boundary_perceptron(p, X, y, f"{name} — [Nama] — [NPM]")

plt.tight_layout()
plt.show()

---
## 🔬 Eksperimen 2: Breaking the Barrier — Multi-Layer Perceptron

**Tujuan:** Memahami bagaimana hidden layer mengatasi limitasi Perceptron tunggal.

> *Perceptron gagal di XOR karena hanya bisa membuat satu garis lurus. Bagaimana jika kita menumpuk beberapa neuron? Inilah ide dari **Multi-Layer Perceptron (MLP)** — arsitektur yang menjadi cikal bakal Deep Learning.*

In [ ]:
# 2A. MLP Menyelesaikan XOR

model_xor = Sequential([
    Dense(4, activation='relu', input_shape=(2,)),
    Dense(1, activation='sigmoid')
])

model_xor.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history_xor = model_xor.fit(X_xor, y_xor, epochs=500, verbose=0)

# Verifikasi hasil
print("Hasil MLP pada XOR:")
for xi, yi in zip(X_xor, y_xor):
    pred = (model_xor.predict(xi.reshape(1, -1), verbose=0) > 0.5).astype(int)[0][0]
    status = '✓' if pred == yi else '✗'
    print(f"  Input: {xi} | Target: {yi} | Prediksi: {pred} | {status}")

In [ ]:
# Visualisasi: MLP vs Perceptron pada XOR
def plot_decision_boundary_keras(model, X, y, title="Decision Boundary"):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))
    grid = np.c_[xx.ravel(), yy.ravel()]

    Z = (model.predict(grid, verbose=0) > 0.5).astype(int).reshape(xx.shape)

    plt.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
    plt.scatter(X[y==0, 0], X[y==0, 1], c='red', marker='o', s=100, edgecolors='k', label='Kelas 0')
    plt.scatter(X[y==1, 0], X[y==1, 1], c='blue', marker='s', s=100, edgecolors='k', label='Kelas 1')
    plt.title(title)
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.legend()
    plt.grid(True, alpha=0.3)

# Side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Perceptron (yang gagal)
plt.sca(axes[0])
p_xor = Perceptron(n_inputs=2)
p_xor.train(X_xor, y_xor, epochs=20)
plot_decision_boundary_perceptron(p_xor, X_xor, y_xor, "Perceptron (GAGAL) — [Nama] — [NPM]")

# MLP (yang berhasil)
plt.sca(axes[1])
plot_decision_boundary_keras(model_xor, X_xor, y_xor, "MLP (BERHASIL) — [Nama] — [NPM]")

plt.tight_layout()
plt.show()

### 2B. MLP pada Fashion-MNIST (First Contact)

In [ ]:
# 2B. MLP sederhana untuk Fashion-MNIST

model_simple = Sequential([
    Dense(64, activation='relu', input_shape=(784,)),
    Dense(10, activation='softmax')
])

model_simple.compile(optimizer='adam',
                     loss='sparse_categorical_crossentropy',
                     metrics=['accuracy'])

history = model_simple.fit(X_train_flat, y_train, epochs=10,
                           validation_split=0.2, verbose=1)

# Evaluasi
test_loss, test_acc = model_simple.evaluate(X_test_flat, y_test, verbose=0)
print(f"\nTest Accuracy: {test_acc*100:.2f}%")

In [ ]:
# Tampilkan 10 gambar yang SALAH diklasifikasi
y_pred = model_simple.predict(X_test_flat, verbose=0).argmax(axis=1)
misclassified_idx = np.where(y_pred != y_test)[0]
selected = np.random.choice(misclassified_idx, min(10, len(misclassified_idx)), replace=False)

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, idx in enumerate(selected):
    ax = axes[i // 5, i % 5]
    ax.imshow(X_test[idx], cmap='gray')
    ax.set_title(f"Pred: {class_names[y_pred[idx]]}\nActual: {class_names[y_test[idx]]}",
                 color='red', fontsize=9)
    ax.axis('off')

plt.suptitle('Gambar yang Salah Diklasifikasi — [Nama] — [NPM]', fontsize=14)
plt.tight_layout()
plt.show()

---
## 🔬 Eksperimen 3: The Tuning Game — Hyperparameter Sensitivity

**Tujuan:** Menganalisis dampak setiap hyperparameter terhadap proses pembelajaran Neural Network.

> *Neural Network memiliki banyak "kenop" yang bisa diputar. Pada eksperimen ini, Anda akan membuktikan sendiri bahwa pemilihan hyperparameter bukan tebak-tebakan — setiap perubahan memiliki dampak yang bisa diprediksi secara logis.*

Gunakan Fashion-MNIST. Untuk setiap sub-eksperimen, ubah **hanya satu variabel** sementara yang lain tetap (*control experiment*).

### 3A. Jumlah Neuron di Hidden Layer

In [ ]:
# Eksperimen 3A: Variasi jumlah neuron
neuron_configs = [8, 32, 128, 512]
results_3a = []

for n_neurons in neuron_configs:
    print(f"\nTraining dengan {n_neurons} neuron...")
    model = Sequential([
        Dense(n_neurons, activation='relu', input_shape=(784,)),
        Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    start = time.time()
    model.fit(X_train_flat, y_train, epochs=10, validation_split=0.2, verbose=0)
    elapsed = time.time() - start

    test_loss, test_acc = model.evaluate(X_test_flat, y_test, verbose=0)
    results_3a.append({'neurons': n_neurons, 'test_acc': test_acc, 'time': elapsed})
    print(f"  -> Test Acc: {test_acc*100:.2f}% | Waktu: {elapsed:.1f}s")

# Tampilkan tabel hasil
print("\n" + "="*55)
print(f"{'Neuron':>10} | {'Test Accuracy':>15} | {'Training Time':>15}")
print("-"*55)
for r in results_3a:
    print(f"{r['neurons']:>10} | {r['test_acc']*100:>14.2f}% | {r['time']:>13.1f}s")
print("="*55)

### 3B. Jumlah Hidden Layer (Depth)

In [ ]:
# Eksperimen 3B: Variasi jumlah hidden layer
layer_configs = [1, 2, 4, 8]
results_3b = []

for n_layers in layer_configs:
    print(f"\nTraining dengan {n_layers} hidden layer(s)...")
    model = Sequential()
    model.add(Dense(64, activation='relu', input_shape=(784,)))
    for _ in range(n_layers - 1):
        model.add(Dense(64, activation='relu'))
    model.add(Dense(10, activation='softmax'))

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    start = time.time()
    model.fit(X_train_flat, y_train, epochs=10, validation_split=0.2, verbose=0)
    elapsed = time.time() - start

    test_loss, test_acc = model.evaluate(X_test_flat, y_test, verbose=0)
    results_3b.append({'layers': n_layers, 'test_acc': test_acc, 'time': elapsed})
    print(f"  -> Test Acc: {test_acc*100:.2f}% | Waktu: {elapsed:.1f}s")

# Tampilkan tabel hasil
print("\n" + "="*55)
print(f"{'Layers':>10} | {'Test Accuracy':>15} | {'Training Time':>15}")
print("-"*55)
for r in results_3b:
    print(f"{r['layers']:>10} | {r['test_acc']*100:>14.2f}% | {r['time']:>13.1f}s")
print("="*55)

### 3C. Learning Rate

In [ ]:
# Eksperimen 3C: Variasi learning rate
learning_rates = [0.0001, 0.001, 0.01, 0.1]
results_3c = []
histories_3c = {}

for lr in learning_rates:
    print(f"\nTraining dengan learning rate = {lr}...")
    model = Sequential([
        Dense(64, activation='relu', input_shape=(784,)),
        Dense(10, activation='softmax')
    ])
    model.compile(optimizer=Adam(learning_rate=lr),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

    history = model.fit(X_train_flat, y_train, epochs=20,
                       validation_split=0.2, verbose=0)
    histories_3c[lr] = history.history['loss']

    test_loss, test_acc = model.evaluate(X_test_flat, y_test, verbose=0)
    results_3c.append({'lr': lr, 'test_acc': test_acc})
    print(f"  -> Test Acc: {test_acc*100:.2f}%")

# Plot training loss untuk semua learning rate
plt.figure(figsize=(10, 5))
for lr, losses in histories_3c.items():
    plt.plot(losses, label=f'LR = {lr}')
plt.xlabel('Epoch')
plt.ylabel('Training Loss')
plt.title('Training Loss per Epoch — Variasi Learning Rate — [Nama] — [NPM]')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Tabel
print("\n" + "="*45)
print(f"{'Learning Rate':>15} | {'Test Accuracy':>15}")
print("-"*45)
for r in results_3c:
    print(f"{r['lr']:>15} | {r['test_acc']*100:>14.2f}%")
print("="*45)

### 3D. Activation Function

In [ ]:
# Eksperimen 3D: Variasi activation function
activations = ['sigmoid', 'tanh', 'relu']
results_3d = []

for act in activations:
    print(f"\nTraining dengan activation = '{act}'...")
    model = Sequential([
        Dense(64, activation=act, input_shape=(784,)),
        Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    start = time.time()
    model.fit(X_train_flat, y_train, epochs=10, validation_split=0.2, verbose=0)
    elapsed = time.time() - start

    test_loss, test_acc = model.evaluate(X_test_flat, y_test, verbose=0)
    results_3d.append({'activation': act, 'test_acc': test_acc, 'time': elapsed})
    print(f"  -> Test Acc: {test_acc*100:.2f}% | Waktu: {elapsed:.1f}s")

# Tabel
print("\n" + "="*55)
print(f"{'Activation':>12} | {'Test Accuracy':>15} | {'Training Time':>15}")
print("-"*55)
for r in results_3d:
    print(f"{r['activation']:>12} | {r['test_acc']*100:>14.2f}% | {r['time']:>13.1f}s")
print("="*55)

---
## 🔬 Eksperimen 4: When More is Less — Overfitting Diagnosis (TAKE-HOME)

**Tujuan:** Mendiagnosis overfitting dan menerapkan teknik regularisasi khas Neural Network.

> *Di Lab 7, Anda pernah melihat k-NN dengan k=1 mendapat akurasi 100% pada training set tapi buruk di test set. Fenomena yang sama terjadi pada Neural Network — dan solusinya lebih canggih: **Dropout** dan **Early Stopping**.*

In [ ]:
# Model A: Sederhana (baseline)
print("Training Model A (Simple)...")
model_A = Sequential([
    Dense(64, activation='relu', input_shape=(784,)),
    Dense(10, activation='softmax')
])
model_A.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_A = model_A.fit(X_train_flat, y_train, epochs=50,
                        validation_split=0.2, verbose=0)
print("Model A selesai!")

In [ ]:
# Model B: Kompleks TANPA regularisasi
print("Training Model B (Complex, no regularization)...")
model_B = Sequential([
    Dense(512, activation='relu', input_shape=(784,)),
    Dense(256, activation='relu'),
    Dense(128, activation='relu'),
    Dense(10, activation='softmax')
])
model_B.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_B = model_B.fit(X_train_flat, y_train, epochs=50,
                        validation_split=0.2, verbose=0)
print("Model B selesai!")

In [ ]:
# Model C: Kompleks DENGAN Dropout + Early Stopping
print("Training Model C (Complex + Dropout + Early Stopping)...")
model_C = Sequential([
    Dense(512, activation='relu', input_shape=(784,)),
    Dropout(0.3),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(10, activation='softmax')
])
model_C.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history_C = model_C.fit(X_train_flat, y_train, epochs=50,
                        validation_split=0.2, verbose=0,
                        callbacks=[early_stop])
print(f"Model C selesai! (Berhenti di epoch {len(history_C.history['loss'])})")

In [ ]:
# Plot Training vs Validation Loss untuk ketiga model
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models_info = [
    ('Model A (Simple)', history_A),
    ('Model B (Complex, No Reg)', history_B),
    ('Model C (Complex + Dropout)', history_C)
]

for i, (name, hist) in enumerate(models_info):
    axes[i].plot(hist.history['loss'], label='Training Loss', linewidth=2)
    axes[i].plot(hist.history['val_loss'], label='Validation Loss', linewidth=2)
    axes[i].set_title(f'{name} — [Nama] — [NPM]', fontsize=11)
    axes[i].set_xlabel('Epoch')
    axes[i].set_ylabel('Loss')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Tabel Perbandingan Performa
print("\n" + "="*75)
print(f"{'Model':<30} | {'Train Acc':>10} | {'Test Acc':>10} | {'Gap':>8}")
print("-"*75)

for name, model, hist in [('A (Simple)', model_A, history_A),
                           ('B (Complex, No Reg)', model_B, history_B),
                           ('C (Complex + Dropout)', model_C, history_C)]:
    train_acc = hist.history['accuracy'][-1]
    test_loss, test_acc = model.evaluate(X_test_flat, y_test, verbose=0)
    gap = train_acc - test_acc
    print(f"{name:<30} | {train_acc*100:>9.2f}% | {test_acc*100:>9.2f}% | {gap*100:>7.2f}%")

print("="*75)

---
## 🔬 Eksperimen 5: Refleksi — Neural Network dalam Konteks AI Pipeline

**Tujuan:** Menghubungkan Neural Network dengan seluruh perjalanan belajar dari Lab 1 hingga Lab 10.

> *Anda telah melewati perjalanan panjang: dari rule-based agent (Lab 1), ke search algorithms (Lab 3), ke Machine Learning klasik (Lab 5-7), lalu ke Computer Vision (Lab 9) dan NLP (Lab 10). Sekarang saatnya melihat ke belakang dan ke depan.*

### 5A. Prediksi Salah yang "Manusiawi"
Gunakan **model terbaik** dari Eksperimen 4 untuk analisis berikut.

In [ ]:
# TODO: Pilih model terbaik dari Eksperimen 4 (ganti model_X sesuai hasil Anda)
best_model = model_C  # Ganti jika model lain lebih baik

# Confusion Matrix
y_pred_best = best_model.predict(X_test_flat, verbose=0).argmax(axis=1)
cm = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap='Blues', xticks_rotation=45)
plt.title('Confusion Matrix — Fashion-MNIST — [Nama] — [NPM]')
plt.tight_layout()
plt.show()

In [ ]:
# Identifikasi 3 pasangan kelas yang paling sering tertukar
# Cari off-diagonal values terbesar di confusion matrix
cm_no_diag = cm.copy()
np.fill_diagonal(cm_no_diag, 0)

print("Top 3 Pasangan Kelas yang Paling Sering Tertukar:")
print("="*60)
for rank in range(3):
    idx = np.unravel_index(cm_no_diag.argmax(), cm_no_diag.shape)
    true_class = class_names[idx[0]]
    pred_class = class_names[idx[1]]
    count = cm_no_diag[idx]
    print(f"  {rank+1}. {true_class} -> salah diprediksi sebagai {pred_class} ({count} kali)")
    cm_no_diag[idx] = 0  # Remove to find next

In [ ]:
# TODO: Tampilkan 5 sampel gambar untuk salah satu pasangan yang tertukar
# Pilih pasangan yang paling sering tertukar dari hasil di atas

# Contoh: jika Shirt sering tertukar dengan T-shirt
true_label_idx = 6   # TODO: Ganti sesuai hasil confusion matrix Anda
pred_label_idx = 0   # TODO: Ganti sesuai hasil confusion matrix Anda

mask = (y_test == true_label_idx) & (y_pred_best == pred_label_idx)
confused_indices = np.where(mask)[0]

if len(confused_indices) > 0:
    selected = confused_indices[:5]
    fig, axes = plt.subplots(1, len(selected), figsize=(12, 3))
    for i, idx in enumerate(selected):
        ax = axes[i] if len(selected) > 1 else axes
        ax.imshow(X_test[idx], cmap='gray')
        ax.set_title(f"Actual: {class_names[true_label_idx]}\nPred: {class_names[pred_label_idx]}",
                     color='red', fontsize=9)
        ax.axis('off')
    plt.suptitle(f'Contoh Kesalahan: {class_names[true_label_idx]} → {class_names[pred_label_idx]} — [Nama] — [NPM]')
    plt.tight_layout()
    plt.show()
else:
    print("Tidak ditemukan kesalahan untuk pasangan ini. Coba pasangan lain.")

### 5B. Perbandingan Pendekatan

Isi tabel di bawah ini berdasarkan pengalaman Anda dari Lab 1 hingga Lab 11:

| Aspek | Rule-Based (Lab 1) | ML Klasik (Lab 6-7) | Neural Network (Lab 11) |
|---|---|---|---|
| Input | Sensor lokal | Fitur yang di-engineer | Pixel mentah / data mentah |
| Cara "belajar" | Aturan ditulis manual | Belajar dari data + fitur | Belajar dari data + fitur otomatis |
| Kekuatan | *isi jawaban Anda* | *isi jawaban Anda* | *isi jawaban Anda* |
| Kelemahan | *isi jawaban Anda* | *isi jawaban Anda* | *isi jawaban Anda* |
| Contoh kasus cocok | *isi jawaban Anda* | *isi jawaban Anda* | *isi jawaban Anda* |

---
## 📝 Catatan

Pastikan Anda menjawab **semua pertanyaan analitik** dari setiap eksperimen di laporan tertulis.  
Laporan ditulis tangan, difoto, dan dikumpulkan sebagai PDF: `1412_Lab11_npm.pdf`